# 01 · ¿Cuánto más cuesta Coca-Cola que Pepsi en la misma tienda?

**Objetivo:** Medir con precios reales cuánto sobreprecio sostiene la marca líder frente a su competidor directo, como caso de valor de marca para directores de marketing y ventas.

**Pregunta:** En presentación equivalente y en la misma tienda, ¿cuánto sobreprecio sostiene Coca-Cola?

Importa porque es la medida más directa del poder de precio de una marca: el mismo cliente, frente al mismo
refrigerador, ve las dos marcas con precios distintos. IC90 = rango probable (bootstrap de tiendas y ciudades).

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, pares, estadistica as E, charts as C
import prima as P
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
pts, st, cat = P.load()
REPS = 500
res_ = {}
for k in P.COMPARA:
    p = P.pairs(pts, st, cat, k)
    tot, _ = P.summary(p, por=(), reps=REPS)
    res_[k] = tot
    p.to_csv(PROC / f"pares_{k}.csv", index=False, encoding="utf-8-sig")
tt = pd.DataFrame(res_).T
display(tt[["n_pares", "claves", "tiendas", "municipios", "brecha_pct", "ic90_inf", "ic90_sup", "brecha_pesos", "pct_invertidos", "pct_iguales"]].round(2))
for k, t in res_.items():
    assert t["ic90_inf"] <= t["brecha_pct"] <= t["ic90_sup"], k

,n_pares,claves,tiendas,municipios,brecha_pct,ic90_inf,ic90_sup,brecha_pesos,pct_invertidos,pct_iguales
lata,184.0,1.0,184.0,56.0,23.68,21.45,24.68,4.50,0.54,1.09
familiar,144.0,1.0,144.0,40.0,4.90,4.90,4.90,-2.00,20.83,0.00
light,36.0,1.0,36.0,18.0,33.21,32.53,37.05,2.73,0.00,0.00


## 1. Lata: precio en pesos

La prima en pesos es la mediana de la diferencia por tienda; también los precios típicos de cada marca.

In [2]:
pl = pd.read_csv(PROC / "pares_lata.csv")
print(f"Precio típico: Coca-Cola ${pl.pesos_b.median():.2f} · Pepsi ${pl.pesos_a.median():.2f} · diferencia típica ${np.median(pl.prima_pesos):.2f}")
display(pl.prima_pct.describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_frame("prima % por tienda"))

Precio típico: Coca-Cola $24.00 · Pepsi $19.00 · diferencia típica $4.50


,prima % por tienda
count,184.0
mean,22.5
std,6.2
min,-5.3
10%,15.8
25%,20.4
50%,23.7
75%,26.3
90%,26.3
max,41.2


## 2. Botella familiar: por litro y por empaque

Por litro Coca-Cola sale algo más cara, pero su botella es más chica (2.75 L contra 3 L): por empaque el cliente paga
menos por la Coca-Cola.

In [3]:
pf = P.pairs(pts, st, cat, "familiar", por="empaque")
tf, _ = P.summary(pf, por=(), reps=REPS)
print(f"Por empaque: Coca-Cola 2.75 L ${pf.pesos_b.median():.2f} · Pepsi 3 L ${pf.pesos_a.median():.2f} · "
      f"prima por empaque {tf['brecha_pct']:+.1f}% (IC90 {tf['ic90_inf']:+.1f} a {tf['ic90_sup']:+.1f})")
inv = (pf.precio_b < pf.precio_a).mean() * 100
print(f"Tiendas donde la botella de Coca-Cola cuesta menos que la de Pepsi: {inv:.0f}%")

Por empaque: Coca-Cola 2.75 L $46.90 · Pepsi 3 L $51.95 · prima por empaque -3.8% (IC90 -3.8 a -3.8)
Tiendas donde la botella de Coca-Cola cuesta menos que la de Pepsi: 79%


## 3. Sensibilidad (lata)

- sin posibles errores ni extremos;
- solo tiendas donde ninguno de los dos precios cambió en la quincena;
- todos los giros (incluye farmacias y conveniencia).

In [4]:
sm = pts.merge(st[st.giro == P.CANAL][["establecimiento_id"]], on="establecimiento_id")
fl = E.flag_prices(sm, sku="sku_id")
clean = sm[~fl.extremo.to_numpy()]
stable = sm[sm.precios_distintos == 1]
var = {"Principal": P.pairs(pts, st, cat, "lata"), "Sin posibles errores ni extremos": P.pairs(clean, st, cat, "lata"),
       "Precio sin cambios en la quincena": P.pairs(stable, st, cat, "lata"), "Todos los giros": P.pairs(pts, st, cat, "lata", canal=None)}
sens = pd.DataFrame({k: P.summary(v, por=(), reps=200)[0] for k, v in var.items()}).T
display(sens[["n_pares", "brecha_pct", "ic90_inf", "ic90_sup", "brecha_pesos", "pct_invertidos"]].round(1))

,n_pares,brecha_pct,ic90_inf,ic90_sup,brecha_pesos,pct_invertidos
Principal,184.0,23.7,21.7,24.3,4.5,0.5
Sin posibles errores ni extremos,184.0,23.7,21.7,24.3,4.5,0.5
Precio sin cambios en la quincena,148.0,24.6,21.7,26.3,4.6,0.7
Todos los giros,210.0,23.7,21.7,25.0,4.5,1.4


In [5]:
tt.to_csv(PROC / "prima_por_comparacion.csv", encoding="utf-8-sig")
sens.to_csv(PROC / "sensibilidad_lata.csv", encoding="utf-8-sig")
F = "01_prima_misma_tienda"
for k, t in res_.items():
    reg(f"prima_{k}", t["brecha_pct"], [t["ic90_inf"], t["ic90_sup"]], n=t["n_pares"], fuente=F)
    reg(f"pesos_{k}", t["brecha_pesos"], n=t["n_pares"], fuente=F)
    reg(f"inv_{k}", t["pct_invertidos"], n=t["n_pares"], fuente=F)
    reg(f"pares_{k}", t["n_pares"], fuente=F); reg(f"mun_{k}", t["municipios"], fuente=F)
reg("lata_coca", pl.pesos_b.median(), fuente=F); reg("lata_pepsi", pl.pesos_a.median(), fuente=F)
reg("lata_p10", pl.prima_pct.quantile(.1), fuente=F); reg("lata_p90", pl.prima_pct.quantile(.9), fuente=F)
reg("fam_coca", pf.pesos_b.median(), fuente=F); reg("fam_pepsi", pf.pesos_a.median(), fuente=F)
reg("fam_empaque", tf["brecha_pct"], [tf["ic90_inf"], tf["ic90_sup"]], n=tf["n_pares"], fuente=F)
reg("fam_coca_mas_barata", inv, fuente=F, nota="% de tiendas donde la botella de Coca-Cola 2.75 L cuesta menos que la Pepsi 3 L")
reg("sens_lata_min", sens.brecha_pct.min(), fuente=F); reg("sens_lata_max", sens.brecha_pct.max(), fuente=F)
t = res_["lata"]
display(Markdown(
    f"**Conclusión.** En la misma tienda, la lata de Coca-Cola cuesta **{t['brecha_pct']:.1f}% más** que la de Pepsi (IC90 "
    f"{t['ic90_inf']:.1f}–{t['ic90_sup']:.1f}%; ${pl.pesos_b.median():.2f} contra ${pl.pesos_a.median():.2f}) y solo en {t['pct_invertidos']:.0f}% de "
    f"{t['n_pares']} tiendas es más barata. En la botella familiar la prima por litro baja a {res_['familiar']['brecha_pct']:.1f}% y por empaque "
    f"la Coca-Cola de 2.75 L cuesta menos que la Pepsi de 3 L en {inv:.0f}% de las tiendas."))

**Conclusión.** En la misma tienda, la lata de Coca-Cola cuesta **23.7% más** que la de Pepsi (IC90 21.4–24.7%; $24.00 contra $19.00) y solo en 1% de 184 tiendas es más barata. En la botella familiar la prima por litro baja a 4.9% y por empaque la Coca-Cola de 2.75 L cuesta menos que la Pepsi de 3 L en 79% de las tiendas.